# Horror Llama 3 8B Detection Experiment

Clean Colab notebook for an English-only horror-text adaptation of *When Detection Fails: The Power of Fine-Tuned Models to Generate Human-Like Social Media Text*.

This notebook expects a prepared local payload:

```text
horror_experiment_payload.zip
├── train_with_descriptions.csv
├── eval_with_descriptions.csv
├── split_metadata.csv
├── corpus_summary.csv
└── corpus_preparation_report.md
```

The GPU part compares:

1. human horror fragments;
2. base `meta-llama/Meta-Llama-3-8B-Instruct` generations;
3. QLoRA fine-tuned Llama 3 8B generations.


## 0. Runtime

Use Google Colab with GPU enabled: `Runtime -> Change runtime type -> T4 GPU` or better.

Llama 3 8B is a gated Hugging Face model. Accept the model license and provide `HF_TOKEN` in Colab secrets/environment, or uncomment `notebook_login()` below.


In [ ]:
#@title 1. Install dependencies
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes sentencepiece scipy scikit-learn pandas matplotlib tqdm evaluate huggingface_hub


In [ ]:
#@title 2. Imports and configuration
from pathlib import Path
import os
import re
import math
import random
import shutil
import zipfile

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

CONFIG = {
    "GEN_MODEL_NAME": "meta-llama/Meta-Llama-3-8B-Instruct",
    "DETECTOR_MODEL_NAME": "openai-community/roberta-base-openai-detector",
    "PAYLOAD_ZIP": "/content/horror_experiment_payload.zip",
    "WORK_DIR": "/content/horror_experiment",
    "SEED": 42,
    "SMOKE_TEST": True,
    "SMOKE_TRAIN_SAMPLES": 80,
    "SMOKE_EVAL_SAMPLES": 24,
    "MAX_TRAIN_SAMPLES": 2000,
    "MAX_EVAL_SAMPLES": 500,
    "BASE_GENERATIONS": 80,
    "MAX_NEW_TOKENS": 450,
    "TEMPERATURE": 0.9,
    "TOP_P": 0.92,
    "LORA_R": 64,
    "LORA_ALPHA": 16,
    "LORA_DROPOUT": 0.05,
    "NUM_EPOCHS": 5,
    "LEARNING_RATE": 2e-4,
    "BATCH_SIZE": 1,
    "GRAD_ACCUM": 8,
    "MAX_SEQ_LENGTH": 1024,
}

random.seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])

WORK_DIR = Path(CONFIG["WORK_DIR"])
DATA_DIR = WORK_DIR / "data"
OUT_DIR = WORK_DIR / "outputs"
MODEL_DIR = WORK_DIR / "models"
for directory in [DATA_DIR, OUT_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Work dir:", WORK_DIR)


In [ ]:
#@title 3. Hugging Face login for gated Llama
from huggingface_hub import login, notebook_login

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in with HF token from environment.")
else:
    print("No HF_TOKEN found. If model loading fails, uncomment and run notebook_login().")
    # notebook_login()


In [ ]:
#@title 4. Upload and unpack prepared payload
from google.colab import files

payload_zip = Path(CONFIG["PAYLOAD_ZIP"])
if not payload_zip.exists():
    print("Upload horror_experiment_payload.zip")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No payload uploaded.")
    first_name = next(iter(uploaded.keys()))
    shutil.move(first_name, payload_zip)

payload_dir = DATA_DIR / "payload"
payload_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(payload_zip, "r") as archive:
    archive.extractall(payload_dir)

print("Payload files:")
for path in sorted(payload_dir.iterdir()):
    print("-", path.name)


In [ ]:
#@title 5. Load train/eval CSVs
train_df = pd.read_csv(payload_dir / "train_with_descriptions.csv")
eval_df = pd.read_csv(payload_dir / "eval_with_descriptions.csv")
split_df = pd.read_csv(payload_dir / "split_metadata.csv")
summary_df = pd.read_csv(payload_dir / "corpus_summary.csv")

if CONFIG["SMOKE_TEST"]:
    train_df = train_df.sample(
        min(CONFIG["SMOKE_TRAIN_SAMPLES"], len(train_df)),
        random_state=CONFIG["SEED"],
    ).reset_index(drop=True)
    eval_df = eval_df.sample(
        min(CONFIG["SMOKE_EVAL_SAMPLES"], len(eval_df)),
        random_state=CONFIG["SEED"],
    ).reset_index(drop=True)
else:
    train_df = train_df.sample(
        min(CONFIG["MAX_TRAIN_SAMPLES"], len(train_df)),
        random_state=CONFIG["SEED"],
    ).reset_index(drop=True)
    eval_df = eval_df.sample(
        min(CONFIG["MAX_EVAL_SAMPLES"], len(eval_df)),
        random_state=CONFIG["SEED"],
    ).reset_index(drop=True)

N_GENERATE = min(CONFIG["BASE_GENERATIONS"], len(eval_df))
if CONFIG["SMOKE_TEST"]:
    N_GENERATE = min(N_GENERATE, CONFIG["SMOKE_EVAL_SAMPLES"])

eval_gen_df = eval_df.head(N_GENERATE).copy()

print("Train chunks:", len(train_df))
print("Eval chunks:", len(eval_df))
print("Generation/evaluation prompts:", len(eval_gen_df))
display(train_df.head(2))


In [ ]:
#@title 6. Load base Llama 3 8B in 4-bit
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(CONFIG["GEN_MODEL_NAME"], use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["GEN_MODEL_NAME"],
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
base_model.config.use_cache = False
print("Loaded:", CONFIG["GEN_MODEL_NAME"])


In [ ]:
#@title 7. Generation helpers
def build_messages(description: str):
    return [
        {
            "role": "system",
            "content": "You write literary horror fragments in English. Write naturally, without explanations.",
        },
        {"role": "user", "content": description},
    ]


def build_prompt(description: str) -> str:
    messages = build_messages(description)
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return "System: " + messages[0]["content"] + "
User: " + description + "
Assistant:"


def clean_generation(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^(Assistant:|assistant:)\s*", "", text).strip()
    return text


def generate_one(model, description: str) -> str:
    prompt = build_prompt(description)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=CONFIG["MAX_NEW_TOKENS"],
            do_sample=True,
            temperature=CONFIG["TEMPERATURE"],
            top_p=CONFIG["TOP_P"],
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return clean_generation(tokenizer.decode(new_tokens, skip_special_tokens=True))


In [ ]:
#@title 8. Generate base AI horror fragments
base_rows = []
for _, row in tqdm(eval_gen_df.iterrows(), total=len(eval_gen_df)):
    generated = generate_one(base_model, row["description"])
    base_rows.append({
        "source_story_id": row["story_id"],
        "source_file": row["source_file"],
        "description": row["description"],
        "text": generated,
        "label": "ai_base",
        "generator": CONFIG["GEN_MODEL_NAME"],
    })

base_ai_df = pd.DataFrame(base_rows)
base_path = DATA_DIR / "ai_base_generations.csv"
base_ai_df.to_csv(base_path, index=False)
display(base_ai_df.head())
print("Saved:", base_path)


In [ ]:
#@title 9. Build SFT dataset
from datasets import Dataset


def format_sft_example(description: str, target_text: str) -> str:
    messages = [
        {
            "role": "system",
            "content": "You write literary horror fragments in English. Write naturally, without explanations.",
        },
        {"role": "user", "content": description},
        {"role": "assistant", "content": target_text},
    ]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return f"System: {messages[0]['content']}
User: {description}
Assistant: {target_text}"

train_texts = [
    format_sft_example(row["description"], row["text"])
    for _, row in train_df.iterrows()
]
train_dataset = Dataset.from_dict({"text": train_texts})
print("SFT examples:", len(train_dataset))
print(train_dataset[0]["text"][:1000])


In [ ]:
#@title 10. QLoRA fine-tuning
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig

model_for_ft = prepare_model_for_kbit_training(base_model)

target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
lora_config = LoraConfig(
    r=CONFIG["LORA_R"],
    lora_alpha=CONFIG["LORA_ALPHA"],
    target_modules=target_modules,
    lora_dropout=CONFIG["LORA_DROPOUT"],
    bias="none",
    task_type="CAUSAL_LM",
)
model_for_ft = get_peft_model(model_for_ft, lora_config)
model_for_ft.print_trainable_parameters()

sft_config = SFTConfig(
    output_dir=str(MODEL_DIR / "horror_llama3_qlora_adapter"),
    num_train_epochs=CONFIG["NUM_EPOCHS"],
    per_device_train_batch_size=CONFIG["BATCH_SIZE"],
    gradient_accumulation_steps=CONFIG["GRAD_ACCUM"],
    learning_rate=CONFIG["LEARNING_RATE"],
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    fp16=True,
    max_seq_length=CONFIG["MAX_SEQ_LENGTH"],
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model_for_ft,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)
trainer.train()

adapter_path = MODEL_DIR / "horror_llama3_qlora_adapter_final"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print("Adapter saved:", adapter_path)


In [ ]:
#@title 11. Generate fine-tuned AI horror fragments
ft_rows = []
for _, row in tqdm(eval_gen_df.iterrows(), total=len(eval_gen_df)):
    generated = generate_one(trainer.model, row["description"])
    ft_rows.append({
        "source_story_id": row["story_id"],
        "source_file": row["source_file"],
        "description": row["description"],
        "text": generated,
        "label": "ai_finetuned",
        "generator": CONFIG["GEN_MODEL_NAME"] + " + QLoRA",
    })

ft_ai_df = pd.DataFrame(ft_rows)
ft_path = DATA_DIR / "ai_finetuned_generations.csv"
ft_ai_df.to_csv(ft_path, index=False)
display(ft_ai_df.head())
print("Saved:", ft_path)


In [ ]:
#@title 12. Stylometric comparison
FEAR_WORDS = {
    "fear", "afraid", "scared", "terror", "horror", "dark", "darkness", "blood", "dead", "death",
    "scream", "screaming", "shadow", "shadows", "night", "nightmare", "ghost", "monster", "creature",
    "door", "window", "whisper", "whispers", "cold", "alone", "body", "grave", "madness", "evil",
}


def extract_features(text: str) -> dict[str, float]:
    words = re.findall(r"[A-Za-z][A-Za-z']*", str(text).lower())
    chars = len(str(text))
    unique_words = len(set(words))
    word_count = len(words)
    return {
        "n_chars": chars,
        "n_words": word_count,
        "type_token_ratio": unique_words / max(1, word_count),
        "avg_word_len": np.mean([len(word) for word in words]) if words else 0,
        "exclamation_count": str(text).count("!"),
        "question_count": str(text).count("?"),
        "ellipsis_count": str(text).count("..."),
        "quote_count": str(text).count('"') + str(text).count("“") + str(text).count("”"),
        "fear_word_rate": sum(1 for word in words if word in FEAR_WORDS) / max(1, word_count),
    }

human_eval_df = eval_gen_df[["text"]].copy()
human_eval_df["label"] = "human"
analysis_df = pd.concat([
    human_eval_df,
    base_ai_df[["text", "label"]],
    ft_ai_df[["text", "label"]],
], ignore_index=True)
features_df = pd.concat([analysis_df, analysis_df["text"].apply(extract_features).apply(pd.Series)], axis=1)
features_path = OUT_DIR / "stylometric_features.csv"
features_df.to_csv(features_path, index=False)
display(features_df.groupby("label").mean(numeric_only=True).round(3))
print("Saved:", features_path)


In [ ]:
#@title 13. Effect sizes: rank-biserial correlation
from scipy.stats import mannwhitneyu

feature_cols = [
    "n_chars", "n_words", "type_token_ratio", "avg_word_len", "exclamation_count",
    "question_count", "ellipsis_count", "quote_count", "fear_word_rate",
]

rows = []
for ai_label in ["ai_base", "ai_finetuned"]:
    for feature in feature_cols:
        human_values = features_df.loc[features_df["label"] == "human", feature].to_numpy()
        ai_values = features_df.loc[features_df["label"] == ai_label, feature].to_numpy()
        u_stat, p_value = mannwhitneyu(human_values, ai_values, alternative="two-sided")
        n1, n2 = len(human_values), len(ai_values)
        rank_biserial = (2 * u_stat) / (n1 * n2) - 1
        rows.append({
            "comparison": f"human_vs_{ai_label}",
            "feature": feature,
            "rank_biserial": rank_biserial,
            "abs_effect": abs(rank_biserial),
            "p_value": p_value,
        })

effects_df = pd.DataFrame(rows).sort_values(["comparison", "abs_effect"], ascending=[True, False])
effects_path = OUT_DIR / "rank_biserial_effects.csv"
effects_df.to_csv(effects_path, index=False)
display(effects_df)
print("Saved:", effects_path)


In [ ]:
#@title 14. Supervised detector: human vs base and human vs fine-tuned
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments


def make_detector_df(human_texts, ai_texts, ai_name):
    human = pd.DataFrame({"text": human_texts, "label": 0, "class_name": "human"})
    ai = pd.DataFrame({"text": ai_texts, "label": 1, "class_name": ai_name})
    return pd.concat([human, ai], ignore_index=True).sample(frac=1, random_state=CONFIG["SEED"]).reset_index(drop=True)


def train_detector(detector_df, run_name):
    train_part, test_part = train_test_split(
        detector_df,
        test_size=0.3,
        random_state=CONFIG["SEED"],
        stratify=detector_df["label"],
    )
    det_tokenizer = AutoTokenizer.from_pretrained(CONFIG["DETECTOR_MODEL_NAME"])
    det_model = AutoModelForSequenceClassification.from_pretrained(CONFIG["DETECTOR_MODEL_NAME"], num_labels=2)

    def tokenize(batch):
        return det_tokenizer(batch["text"], truncation=True, max_length=512)

    train_ds = Dataset.from_pandas(train_part[["text", "label"]]).map(tokenize, batched=True)
    test_ds = Dataset.from_pandas(test_part[["text", "label"]]).map(tokenize, batched=True)
    collator = DataCollatorWithPadding(tokenizer=det_tokenizer)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
        precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
        metrics = {
            "accuracy": accuracy_score(labels, preds),
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }
        try:
            metrics["roc_auc"] = roc_auc_score(labels, probs)
        except Exception:
            metrics["roc_auc"] = float("nan")
        return metrics

    args = TrainingArguments(
        output_dir=str(MODEL_DIR / f"detector_{run_name}"),
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=1 if CONFIG["SMOKE_TEST"] else 3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        report_to="none",
    )
    detector_trainer = Trainer(
        model=det_model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        tokenizer=det_tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )
    detector_trainer.train()
    return detector_trainer.evaluate()

human_texts = eval_gen_df["text"].tolist()
base_detector_df = make_detector_df(human_texts, base_ai_df["text"].tolist(), "ai_base")
ft_detector_df = make_detector_df(human_texts, ft_ai_df["text"].tolist(), "ai_finetuned")

base_metrics = train_detector(base_detector_df, "human_vs_base_ai")
ft_metrics = train_detector(ft_detector_df, "human_vs_finetuned_ai")

summary = pd.DataFrame([
    {"scenario": "human vs base AI", **{k.replace("eval_", ""): v for k, v in base_metrics.items() if k.startswith("eval_")}},
    {"scenario": "human vs fine-tuned AI", **{k.replace("eval_", ""): v for k, v in ft_metrics.items() if k.startswith("eval_")}},
])
summary_path = OUT_DIR / "detector_summary.csv"
summary.to_csv(summary_path, index=False)
display(summary)
print("Saved:", summary_path)


In [ ]:
#@title 15. Export results
zip_out = "/content/horror_experiment_results"
shutil.make_archive(zip_out, "zip", WORK_DIR)
print("Archive created:", zip_out + ".zip")
files.download(zip_out + ".zip")
